# Test a RAG retriever in an LLM model

In [7]:
import os, sys

from vertexai import rag
import vertexai
from vertexai.generative_models import GenerativeModel, Tool


utils_path = "../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication

# Set environment variables
dotenv_path = "../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()

# Initialize Vertex AI API once per session
vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
              location=os.environ["GOOGLE_CLOUD_LOCATION"],
              staging_bucket=os.environ["STAGING_BUCKET"])


## Get the corpa

In [4]:
# existing_corpora = rag.list_corpora()
existing_corpora = [c.name for c in rag.list_corpora()]
existing_corpora


['projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/4611686018427387904']

In [6]:

# Retrieve an existing RAG corpus
# rag_corpus = rag.get_corpus(name="projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/1290281293241647104")
rag_corpus = rag.get_corpus(name=existing_corpora[0])

q1 = "What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites"
q2= "What are LEAP Exams?"


# Direct context retrieval
rag_retrieval_config = rag.RagRetrievalConfig(
    top_k=3,  # Optional
    filter=rag.Filter(vector_distance_threshold=0.5),  # Optional
)

for query in [q1, q2]:
    response = rag.retrieval_query(
        rag_resources=[
            rag.RagResource(
                rag_corpus=rag_corpus.name,
                # Optional: supply IDs from `rag.list_files()`.
                # rag_file_ids=["rag-file-1", "rag-file-2", ...],
            )
        ],
        text=q2,
        rag_retrieval_config=rag_retrieval_config,
    )
    print(response)
    print()




contexts {
  contexts {
    source_uri: "gs://ccc-crawl_data_xb/crawl_data/text_files/wwwecsorg_2025May01_text.txt"
    source_display_name: "wwwecsorg_2025May01_text.txt"
    text: "All rights reserved. State Information Request: Assessment Approaches: Policy Priority Areas Resources by State Trending Topics Ed Note Blog Education Commission of the States is the trusted source for comprehensive knowledge and unbiased resources on education policy issues ranging from early learning through postsecondary education. Subscribe to our publications and stay informed. Need more information? Contact one of our policy experts. A state education agency official asked for information on assessment approaches beyond end-of-year assessments. Our response includes information on college entrance exams (that are used as assessments); through-course assessments; computer-adaptive tests; approaches that include multiple, distinct summative assessments throughout the school year; and performance task a

## Create a model instance calling a tool

In [8]:

# Enhance generation
# Create a RAG retrieval tool
rag_retrieval_tool = Tool.from_retrieval(
    retrieval=rag.Retrieval(
        source=rag.VertexRagStore(
            rag_resources=[
                rag.RagResource(
                    rag_corpus=rag_corpus.name,  # Currently only 1 corpus is allowed.
                    # Optional: supply IDs from `ccc_agent_workflow.list_files()`.
                    # rag_file_ids=["ccc_agent_workflow-file-1", "ccc_agent_workflow-file-2", ...],
                )
            ],
            rag_retrieval_config=rag_retrieval_config,
        ),
    )
)

In [9]:
# Create a Gemini model instance
rag_model = GenerativeModel(
    model_name="gemini-2.0-flash-001", tools=[rag_retrieval_tool]
)

# Generate response
response = rag_model.generate_content(q1)
print(response.text)


The California Community Colleges Chancellor’s Office (CCCCO) is committed to ensuring digital accessibility for people with disabilities by:

*   Continually improving the user experience and applying relevant accessibility standards.
*   Following the Web Content Accessibility Guidelines (WCAG).
*   Supporting present-day technical accessibility standards, such as ARIA 1.2.
*   Using an accessibility partner for external evaluation.
*   Working to ensure all content on their websites is accessible to everyone.
*   Having accessibility reports and a Voluntary Product Accessibility Template (VPAT) plan on file for each of their websites.

As of July 2023, the CCCCO websites are partially conformant with WCAG 2.1 level AA. They welcome feedback on the accessibility of their websites, and VPATs are available upon request.


In [10]:
# Generate response
response = rag_model.generate_content(q2)
print(response.text)


LEAP Exams are Limited Examination Appointment Program Exams, which are alternate examination and appointment processes for recruiting and hiring individuals with disabilities into State service. Please contact the Department of Rehabilitation for LEAP certification.

